# Classical Word Embeddings — Pre-trained Word2Vec & Average Word2Vec

In the previous notebook, we trained a **custom Word2Vec model** from scratch on a small story corpus. Here we use a **pre-trained Word2Vec model** trained by Google on a massive real-world dataset — and then explore how to extend word-level embeddings to **sentence-level vectors** using Average Word2Vec.

---

## Part A: Pre-trained Word2Vec (Google News)

Training Word2Vec from scratch requires **huge amounts of text**. Google released a model trained on **3 billion words** from Google News articles:

| Property | Value |
|---|---|
| Training data | Google News (~3B words) |
| Vocabulary size | ~3 million unique words |
| Vector dimensions | 300 |
| Architecture | Skip-Gram |

Because it was trained on news data, it understands **real-world entities**, **proper nouns**, and **domain-specific language** very well.

---

## Part B: Average Word2Vec

Word2Vec produces one vector **per word**. But what if we need one vector **per sentence or document**?

**Average Word2Vec** solves this by:
1. Looking up the vector for each word in the sentence.
2. Taking the **element-wise mean** across all word vectors.
3. Returning a single vector of the same dimensionality.

```
"I love machine learning"
        ↓
[ vec(i) + vec(love) + vec(machine) + vec(learning) ] / 4
        ↓
  one 300-dim sentence vector
```

> **Note:** Average Word2Vec is a classical technique. Modern state-of-the-art (SOTA) approaches use **transformer-based models** (BERT, GPT) that produce context-aware sentence embeddings.

## Step 1: Import Libraries

- **`Word2Vec`** — class for training custom models (not used here directly, but commonly imported together).
- **`KeyedVectors`** — lightweight container that holds **only the word vectors** without the training graph. Pre-trained models are often distributed as `KeyedVectors` to save memory.
- **`gensim.downloader`** — Gensim's built-in API to download and cache pre-trained models and datasets directly.

In [ ]:
from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api
import numpy as np

## Step 2: Load the Pre-trained Google News Word2Vec Model

`api.load("word2vec-google-news-300")` downloads (first time only, ~1.6 GB) and loads the Google News Word2Vec model.

Key facts about this model:
- Trained using the **Skip-Gram** architecture (`sg=1`).
- Each word is represented as a **300-dimensional vector**.
- Trained on **Google News articles** — so it understands news-domain language well (crime, politics, sports, finance).
- Words are **case-sensitive** — `'Sunny'` and `'sunny'` are different entries.

> **First run:** This will download the model file. Subsequent runs load it from local cache instantly.

In [ ]:
# Skip-Gram model trained on Google News (~3B words, 300 dimensions)
model = api.load("word2vec-google-news-300")
print("Model loaded successfully!")

## Step 3: Look Up a Word Vector

`model["word"]` returns the **300-dimensional embedding vector** for the word as a NumPy array.

The numbers themselves are not human-readable, but their **relative positions** in 300D space encode rich semantic meaning learned from billions of news sentences.

In [ ]:
print("Vector for 'sunny':")
print(model["sunny"])
print("\nVector dimensions:", len(model["sunny"]))

## Step 4: Word2Vec Encodes Single Words — Not Phrases

Word2Vec maps **individual tokens** to vectors. Passing a multi-word string like `"i am sunny"` will raise a `KeyError` because the model has no entry for that exact string as a single key.

To represent a phrase or sentence, you must encode each word **separately** and then combine them — which is exactly what **Average Word2Vec** does in Part B.

> The cell below is intentionally expected to raise a `KeyError` — it demonstrates this important limitation.

In [ ]:
# This will raise a KeyError — Word2Vec only handles single word tokens
try:
    model["i am sunny"]
except KeyError as e:
    print(f"KeyError: {e}")
    print("Word2Vec cannot encode multi-word phrases as a single lookup.")

## Step 5: Find Most Similar Words

`.most_similar("word")` returns the **top 10 nearest neighbors** in the 300-dimensional vector space, ranked by **cosine similarity**.

Because this model was trained on billions of news sentences, the results are much richer and more accurate than our small custom model. Words like `'cricket'` will return cricket-related terms from real news coverage.

In [ ]:
print("Most similar to 'man':")
print(model.most_similar("man"))

In [ ]:
print("Most similar to 'cricket':")
print(model.most_similar("cricket"))

## Step 6: Measure Similarity Between Two Words

`.similarity("word1", "word2")` computes the **cosine similarity** between two word vectors and returns a float between `-1` and `1`.

| Score | Meaning |
|---|---|
| `1.0` | Identical words (same vector) |
| `0.7 – 0.9` | Strongly related |
| `0.3 – 0.6` | Loosely related |
| `~0.0` | Unrelated |
| Negative | Opposite contexts |

Notice:
- `man` vs `woman` → high score (same context, different gender)
- `man` vs `python` → low score (completely different domains)
- `man` vs `man` → exactly `1.0` (identical vector)
- `happy` vs `joy` → high score (synonyms)
- `python` vs `java` → moderate score (both programming languages)
- `sunny` vs `Sunny` → different scores! (case-sensitive vocabulary)

In [ ]:
print(f"man  ↔ woman  : {model.similarity('man', 'woman'):.4f}")
print(f"man  ↔ python : {model.similarity('man', 'python'):.4f}")
print(f"man  ↔ man    : {model.similarity('man', 'man'):.4f}")
print(f"happy↔ joy    : {model.similarity('happy', 'joy'):.4f}")
print(f"python↔ java  : {model.similarity('python', 'java'):.4f}")

## Step 7: Case Sensitivity in the Vocabulary

The Google News model is **case-sensitive** because proper nouns in news (e.g., `'Sunny'` as a person's name) carry different meaning from common adjectives (`'sunny'` weather).

This demonstrates that `'sunny'` and `'Sunny'` are **two separate entries** in the vocabulary with different vectors.

In [ ]:
print(f"sunny ↔ sunny  : {model.similarity('sunny', 'sunny'):.4f}")
print(f"sunny ↔ Sunny  : {model.similarity('sunny', 'Sunny'):.4f}")
print("\n'sunny' (lowercase) = weather/adjective context")
print("'Sunny' (uppercase) = proper noun / person name context")

## Step 8: Find the Odd Word Out

`.doesnt_match([list])` identifies the word whose vector is **farthest from the group's mean vector**.

Here we mix programming languages with an animal — the model should correctly identify `'DOG'` as the odd one out, because `'PHP'`, `'JAVA'`, and `'C++'` all cluster together in the programming domain.

In [ ]:
result = model.doesnt_match(["PHP", "JAVA", "DOG", "C++"])
print(f"Odd word out from ['PHP', 'JAVA', 'DOG', 'C++']: {result}")

## Step 9: Word Vector Arithmetic — King − Man + Woman = Queen

This is the most famous demonstration of Word2Vec's power. The model encodes **semantic relationships as directions** in vector space:

$$\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$$

The intuition:
- The vector from `man → king` encodes the concept of **"royalty for males"**.
- Applying the same direction to `woman` gives the **"royalty for females"** position.
- The nearest word to that position in the vocabulary is `queen`.

We compute the arithmetic result vector and then use `.most_similar([vec])` to find the closest real word to it.

In [ ]:
# king - man + woman ≈ queen
vec = model['king'] - model['man'] + model['woman']
print("Top words closest to (king - man + woman):")
print(model.most_similar([vec]))

## Part B — Average Word2Vec

---

## Step 10: The Problem — Word2Vec is Word-Level

Word2Vec gives us one vector **per word**. In many real-world tasks (sentiment analysis, document classification, semantic search), we need one vector **per sentence or document**.

**Average Word2Vec** is the simplest approach:

1. **Tokenize** the sentence into words.
2. **Look up** the Word2Vec vector for each word.
3. **Average** all the word vectors element-by-element.
4. The result is a single vector of the same dimensionality (300 here).

We start by defining a sentence and tokenizing it.

In [ ]:
sentence = "I love machine learning"
words = sentence.lower().split()
print("Tokens:", words)

## Step 11: Retrieve a Vector for Each Word

For each token in the sentence, we look up its 300-dimensional Word2Vec vector and collect them into a list.

After this step, `word_vectors` is a **list of 4 arrays**, each of shape `(300,)` — one per word in the sentence.

In [ ]:
word_vectors = []
for word in words:
    print(f"  Looking up vector for: '{word}'")
    word_vectors.append(model[word])

print(f"\nCollected {len(word_vectors)} word vectors, each of shape: {word_vectors[0].shape}")

## Step 12: Compute the Sentence Vector (Average)

`np.mean(word_vectors, axis=0)` computes the **element-wise mean** across all word vectors:

- `axis=0` means we average **across rows** (across words), not across dimensions.
- Input shape: `(4, 300)` — 4 words × 300 dimensions.
- Output shape: `(300,)` — one averaged sentence vector.

Each of the 300 output values is the average of the 4 corresponding word vector values at that position.

In [ ]:
sentence_vector = np.mean(word_vectors, axis=0)
print("Sentence vector shape:", sentence_vector.shape)
print("\nSentence vector (first 20 values):")
print(sentence_vector[:20])

## Step 13: Verify the Output Shape

Regardless of how many words are in the sentence, the output of Average Word2Vec is always a **fixed-size vector** matching the model's `vector_size`.

This is important for ML pipelines — all inputs to a classifier or similarity engine must have the **same shape**.

In [ ]:
print(f"Input sentence  : '{sentence}'")
print(f"Number of words : {len(words)}")
print(f"Output shape    : {sentence_vector.shape}")
print("\nNo matter how long the sentence is, the output is always shape (300,)")

## Summary

### Part A — Pre-trained Word2Vec

| Step | Action | Purpose |
|------|--------|---------|
| 1 | Import libraries | Load `Word2Vec`, `KeyedVectors`, `gensim.downloader` |
| 2 | Load pre-trained model | Download Google News Word2Vec (300D, Skip-Gram) |
| 3 | Look up a word vector | Retrieve the 300-dim vector for a single word |
| 4 | Phrase lookup fails | Demonstrate that Word2Vec is word-level only |
| 5 | Find similar words | Query nearest neighbors using cosine similarity |
| 6 | Measure word similarity | Compare pairs of words with a similarity score |
| 7 | Case sensitivity | Show that `sunny` ≠ `Sunny` in the vocabulary |
| 8 | Find odd word out | Identify the word that doesn't belong in a group |
| 9 | Vector arithmetic | king − man + woman ≈ queen |

### Part B — Average Word2Vec

| Step | Action | Purpose |
|------|--------|---------|
| 10 | Tokenize sentence | Split sentence into lowercase word tokens |
| 11 | Retrieve word vectors | Look up the Word2Vec vector for each token |
| 12 | Compute average | `np.mean(word_vectors, axis=0)` → one sentence vector |
| 13 | Verify output shape | Confirm fixed-size `(300,)` output regardless of sentence length |

---

## Classical Embeddings — Where They Stand

| Technique | Unit | Captures Frequency | Captures Semantics | Context-Aware |
|---|---|---|---|---|
| One Hot Encoding | word | ❌ | ❌ | ❌ |
| Bag of Words | document | ✅ | ❌ | ❌ |
| TF-IDF | document | ✅ (weighted) | ❌ | ❌ |
| Word2Vec | word | ✅ | ✅ | ❌ (static) |
| Avg Word2Vec | sentence | ✅ | ✅ | ❌ (static) |
| **BERT / GPT** | sentence | ✅ | ✅ | **✅** |

> **Limitation of Average Word2Vec:** All words contribute equally to the sentence vector. Word order and grammar are lost. The word `'not'` in `"I do not like this"` vs `"I like this"` would barely change the sentence vector. This limitation is addressed by **transformer-based models** (BERT, GPT) covered in the next class.